# 🚨 DevOps Incident Triage — SRE Agent Training

**OpenEnv Hackathon 2026 · Theme 3.1 — Professional World Modelling**

This notebook trains a Llama-3.2-1B-Instruct model to act as an on-call SRE.
The agent must investigate production incidents, apply safe mitigations,
communicate status, and resolve incidents through a typed action interface.

**Pipeline:**
1. **SFT Warm-start** — 1 epoch on optimal action sequences (all 6 tasks)
2. **GRPO RL Training** — 200 steps with 4 reward signals
3. **Reward curve plot** — saved as `reward_curve.png`
4. **Before/After comparison** — base vs trained scores across all 6 tasks

**Prerequisites:** Select **Runtime → Change runtime type → T4 GPU** before running.

---

In [ ]:
# Cell 1 — GPU check (must be T4 or better)
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:', result.stdout.strip())
else:
    raise RuntimeError('❌ No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2 — Install dependencies
# Unsloth must be installed first; the rest follow
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl>=0.12 datasets pydantic pyyaml fastapi uvicorn openai httpx pandas matplotlib -q
print('✅ Dependencies installed')

In [ ]:
# Cell 3 — Clone the environment repo and add to path
import os, sys

REPO_URL = 'https://github.com/Christy-saji/incident-ops-openenv'
REPO_DIR = '/content/incident-ops-openenv'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
!git log --oneline -5
print('✅ Repo ready at', REPO_DIR)

In [ ]:
# Cell 4 — Smoke-test environment (no GPU needed)
from env.environment import DevOpsEnv
from graders.grader import compute_score
from tasks.task_config import TASK_CONFIGS

print('Testing all 6 task scenarios...')
for task in TASK_CONFIGS:
    env = DevOpsEnv(task=task)
    obs = env.reset()
    _, r, done, info = env.step('acknowledge_incident')
    score, breakdown = compute_score(task, env._state)
    print(f'  [{task:12s}] step_reward={r:+.2f}  score={score:.2f}  phase={obs["incident_phase"]}')

# Test partial observability
env = DevOpsEnv(task='hard', partial_obs=True)
obs = env.reset()
assert obs['known_findings'] == ['[hidden — partial observability active]'], 'Partial obs broken!'

# Test stochastic mode
env_s = DevOpsEnv(task='easy', stochastic=True)
env_s.reset()
env_s.step('inspect_deploy_history')  # triggers noise

print('\n✅ Environment smoke test PASSED (6 tasks, partial_obs, stochastic)')

In [ ]:
# Cell 5 — Smoke-test reward functions (no GPU)
import json
from train import (
    format_reward_func,
    step_reward_func,
    anti_cheat_reward_func,
    task_alignment_reward_func,
    generate_prompts,
    generate_sft_dataset,
)
from env.models import VALID_ACTIONS

# format reward
assert format_reward_func([], [[{'content': 'no_op'}]]) == [1.0], 'format_reward: valid action failed'
assert format_reward_func([], [[{'content': 'banana'}]]) == [0.0], 'format_reward: invalid action failed'
print('  ✅ format_reward_func')

# anti-cheat
assert anti_cheat_reward_func([], [[{'content': 'no_op'}]]) == [-0.2], 'anti_cheat: no_op failed'
assert anti_cheat_reward_func([], [[{'content': 'rollback_auth_deploy'}]]) == [0.1], 'anti_cheat: good action failed'
print('  ✅ anti_cheat_reward_func')

# task alignment
env = DevOpsEnv(task='easy')
state = env.reset()
prompts = [[
    {'role': 'system', 'content': 'sys'},
    {'role': 'user',   'content': json.dumps(state)},
]]
r = task_alignment_reward_func(prompts, [[{'content': 'inspect_deploy_history'}]])
assert r == [0.3], f'task_alignment required diagnostic: expected [0.3], got {r}'
print('  ✅ task_alignment_reward_func')

# step_reward mid-episode replay
env2 = DevOpsEnv(task='network')
state2 = env2.reset()
# simulate 1 step already taken
state2['recent_actions'] = ['acknowledge_incident']
prompts2 = [[
    {'role': 'system', 'content': 'sys'},
    {'role': 'user',   'content': json.dumps(state2)},
]]
r2 = step_reward_func(prompts2, [[{'content': 'inspect_network_topology'}]])
assert len(r2) == 1, 'step_reward: wrong output length'
print('  ✅ step_reward_func (mid-episode replay)')

# curriculum dataset
ds = generate_prompts(per_task_n=3, mid_episode_n=3)
tasks_seen = {json.loads(item['prompt'][-1]['content'])['task'] for item in ds}
assert tasks_seen == set(['easy', 'medium', 'hard', 'network', 'memory_leak', 'disk_full']), \
    f'Missing tasks in dataset: {tasks_seen}'
print(f'  ✅ generate_prompts: {len(ds)} prompts across {len(tasks_seen)} tasks')

# SFT dataset
sft_ds = generate_sft_dataset()
print(f'  ✅ generate_sft_dataset: {len(sft_ds)} (state, optimal_action) pairs')

print('\n✅ All reward function tests PASSED')

In [ ]:
# Cell 6 — SANITY TRAIN (5 steps · ~2 min · proves the pipeline runs)
# Run this BEFORE the full train to catch any import/config errors cheaply.
import gc, torch
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import GRPOTrainer, GRPOConfig
from train import (
    format_reward_func, step_reward_func,
    anti_cheat_reward_func, task_alignment_reward_func,
    generate_prompts, RewardLoggerCallback,
)

MODEL_ID = 'unsloth/Llama-3.2-1B-Instruct'
MAX_SEQ  = 512

PatchDPOTrainer()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model, r=8,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    use_gradient_checkpointing='unsloth',
)

dataset = generate_prompts(per_task_n=2, mid_episode_n=2)

sanity_args = GRPOConfig(
    output_dir='outputs_sanity',
    learning_rate=1e-5,
    max_steps=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    logging_steps=1,
    num_generations=2,
    max_prompt_length=256,
    max_completion_length=16,
)

sanity_cb = RewardLoggerCallback(log_path='outputs_sanity/reward_log.csv')
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward_func, step_reward_func,
                  anti_cheat_reward_func, task_alignment_reward_func],
    args=sanity_args,
    train_dataset=dataset,
    callbacks=[sanity_cb],
)

trainer.train()

import os, pandas as pd
assert os.path.exists('outputs_sanity/reward_log.csv'), 'reward_log.csv not created!'
df = pd.read_csv('outputs_sanity/reward_log.csv')
print('\nSanity reward log:')
print(df.to_string())

# Clean up to free VRAM before full train
del model, tokenizer, trainer
gc.collect()
torch.cuda.empty_cache()
print('\n✅ SANITY TRAIN PASSED — pipeline is working. Proceed to full train.')

In [ ]:
# Cell 7 — FULL TRAINING (SFT warm-start → GRPO 200 steps · ~30–45 min on T4)
# This cell runs train.py's main() which does:
#   Phase 1: SFT warm-start on all 6 optimal trajectories
#   Phase 2: GRPO RL with 4 reward signals for 200 steps

import gc, torch

# Clear any leftover state from the sanity run
gc.collect()
torch.cuda.empty_cache()

# Run the full training pipeline
%run train.py

import os
assert os.path.exists('outputs_grpo/reward_log.csv'), 'Training did not produce reward_log.csv'
import pandas as pd
df = pd.read_csv('outputs_grpo/reward_log.csv')
print(f'\nTraining steps logged: {len(df)}')
print(df.tail(10).to_string())
print('\n✅ Full training complete!')

In [ ]:
# Cell 8 — Plot reward curves and save as PNG
# This is the key evidence artefact for the hackathon judges.
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

df = pd.read_csv('outputs_grpo/reward_log.csv')

reward_cols = [
    ('reward',                           'Overall Reward',          '#6366f1'),
    ('reward_format_reward_func',        'Format Reward',           '#22c55e'),
    ('reward_step_reward_func',          'Step Reward',             '#f59e0b'),
    ('reward_task_alignment_reward_func','Task Alignment Reward',   '#ec4899'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(
    'GRPO Training — DevOps Incident Triage SRE Agent\n'
    'Llama-3.2-1B-Instruct + SFT warm-start + GRPO (4 reward signals)',
    fontsize=13, fontweight='bold'
)

def smooth(series, w=10):
    """Simple moving average for cleaner curve visualisation."""
    return series.rolling(window=w, min_periods=1).mean()

for (col, title, color), ax in zip(reward_cols, axes.flat):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    series = pd.to_numeric(df[col], errors='coerce').dropna()
    steps  = series.index
    ax.plot(steps, series.values, alpha=0.3, color=color, linewidth=1, label='raw')
    ax.plot(steps, smooth(series).values, color=color, linewidth=2, label='smoothed (w=10)')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Reward')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    # Add trend line
    z = np.polyfit(steps, series.values, 1)
    p = np.poly1d(z)
    ax.plot(steps, p(steps), '--', color='grey', linewidth=1, alpha=0.7, label='trend')

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Reward curve saved to reward_curve.png')

# Print summary statistics
overall = pd.to_numeric(df['reward'], errors='coerce').dropna()
first10 = overall.head(10).mean()
last10  = overall.tail(10).mean()
print(f'\nOverall reward — first 10 steps avg: {first10:.4f}')
print(f'Overall reward — last  10 steps avg: {last10:.4f}')
print(f'Improvement: {last10 - first10:+.4f}')

In [ ]:
# Cell 9 — Before/After comparison: base model vs trained model (all 6 tasks)
!python compare_inference.py \
    --base-model unsloth/Llama-3.2-1B-Instruct \
    --trained-model ./trained_sre_agent

In [ ]:
# Cell 10 — Save reward_curve.png to Google Drive (optional)
# Uncomment the block below if you want to persist the plot to Drive.

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# dest = '/content/drive/MyDrive/devops_sre_results/'
# os.makedirs(dest, exist_ok=True)
# shutil.copy('reward_curve.png', dest + 'reward_curve.png')
# shutil.copy('outputs_grpo/reward_log.csv', dest + 'reward_log.csv')
# print('✅ Results saved to Google Drive')

# --- Download directly to your machine ---
from google.colab import files
files.download('reward_curve.png')
files.download('outputs_grpo/reward_log.csv')
print('✅ Files sent to browser download')

In [ ]:
# Cell 11 — Push trained model to HuggingFace Hub (optional)
# Set HF_TOKEN before running.

HF_TOKEN    = ''   # ← paste your HF write token here
HF_REPO_ID  = 'chritsysajii/sre-agent-llama3-grpo'  # ← your HF repo

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

    # Load the merged model that train.py saved
    from unsloth import FastLanguageModel
    model_push, tok_push = FastLanguageModel.from_pretrained(
        model_name='./trained_sre_agent',
        max_seq_length=1024, dtype=None, load_in_4bit=False,
    )
    model_push.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tok_push.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print(f'✅ Model pushed to https://huggingface.co/{HF_REPO_ID}')
else:
    print('ℹ️  Skipped: set HF_TOKEN to push the trained model to the Hub.')

In [ ]:
# Cell 12 — Test the FastAPI server (optional, for Space validation)
import subprocess, time, httpx, json

proc = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '7860'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(4)

base = 'http://localhost:7860'

# Health
r = httpx.get(f'{base}/health', timeout=10); print('health:', r.json())

# Tasks
r = httpx.get(f'{base}/tasks', timeout=10)
print('tasks:', [t['name'] for t in r.json()['tasks']])

# Reset
r = httpx.post(f'{base}/reset', json={'task': 'network', 'session_id': 'test'}, timeout=10)
print('reset:', r.json()['observation']['incident_title'])

# Step
r = httpx.post(f'{base}/step', json={'name': 'inspect_network_topology', 'session_id': 'test'}, timeout=10)
print('step reward:', r.json()['reward'])

# Demo endpoint
r = httpx.get(f'{base}/demo?task=easy', timeout=30)
demo = r.json()
print(f'demo: task={demo["task"]} resolved={demo["resolved"]} final_score={demo["final_score"]}')

# Leaderboard
r = httpx.get(f'{base}/leaderboard', timeout=10); print('leaderboard:', r.json())

proc.terminate()
print('\n✅ FastAPI server test PASSED')